# Phase III – Part A: Faber Baseline Strategy
Implements Faber's 10-month SMA momentum rule:
- **Buy** when month-end price > 10-month SMA
- **Move to cash** when month-end price < 10-month SMA
- Equal weight across all invested ETFs
- Monthly rebalancing from February 2007 to December 2024

In [1]:
import pandas as pd
import numpy as np

ETFS       = ['SPY', 'EFA', 'TLT', 'VNQ', 'DBC']
BACKTEST_START = '2010-09-30'
BACKTEST_END   = '2024-08-31'

prices = pd.read_csv('cleaned_prices.csv', index_col=0, parse_dates=True)
print(f'Loaded {len(prices)} month-end observations.')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')

Loaded 227 month-end observations.
Date range: 2006-02-28 to 2024-12-31


## Step 1: Calculate 10-Month SMA
For each ETF at each month-end, compute the simple moving average of the last 10 monthly closing prices.

In [2]:
# rolling(10) uses only past prices — no lookahead bias
sma_10 = prices[ETFS].rolling(10).mean()

print('10-month SMA calculated.')
print(f'First valid SMA date: {sma_10.dropna().index[0].date()}')

10-month SMA calculated.
First valid SMA date: 2006-11-30


## Step 2: Generate Buy/Sell Signals
Signal = 1 (invest) if price > SMA, else 0 (cash).

In [3]:
signals = (prices[ETFS] > sma_10).astype(int)

# Restrict to backtest period
signals = signals.loc[BACKTEST_START:BACKTEST_END]
prices_bt = prices[ETFS].loc[BACKTEST_START:BACKTEST_END]

print(f'Backtest period: {signals.index[0].date()} to {signals.index[-1].date()}')
print(f'{len(signals)} monthly observations.')
signals.head()

Backtest period: 2010-09-30 to 2024-08-31
168 monthly observations.


,SPY,EFA,TLT,VNQ,DBC
2010-09-30,1,1,1,1,1
2010-10-31,1,1,1,1,1
2010-11-30,1,1,1,1,1
2010-12-31,1,1,0,1,1
2011-01-31,1,1,0,1,1


## Step 3: Run Monthly Rebalancing Loop
At each month-end:
- Identify which ETFs have price > 10-month SMA
- Assign equal weight to each invested ETF
- If no ETFs qualify, portfolio sits in cash (0% return)
- Record the weighted portfolio return for that month

In [4]:
monthly_returns = prices[ETFS].pct_change()
monthly_returns = monthly_returns.loc[BACKTEST_START:BACKTEST_END]

portfolio_returns = []

for date in signals.index:
    invested = signals.loc[date]
    n_invested = invested.sum()

    if n_invested == 0:
        # All ETFs below SMA — sit in cash
        portfolio_returns.append({'date': date, 'return': 0.0, 'n_invested': 0})
    else:
        # Equal weight across invested ETFs
        weight = 1.0 / n_invested
        ret = (monthly_returns.loc[date] * invested * weight).sum()
        portfolio_returns.append({'date': date, 'return': ret, 'n_invested': int(n_invested)})

faber_returns = pd.DataFrame(portfolio_returns).set_index('date')

print(f'Backtest complete: {len(faber_returns)} monthly observations.')
print(f'Months fully in cash: {(faber_returns["n_invested"] == 0).sum()}')
faber_returns.head(10)

Backtest complete: 168 monthly observations.
Months fully in cash: 4


,return,n_invested
date,,
2010-09-30,0.058963,5
2010-10-31,0.024674,5
2010-11-30,-0.017603,5
2010-12-31,0.073591,4
2011-01-31,0.028084,4
2011-02-28,0.039749,4
2011-03-31,-0.003322,4
2011-04-30,0.047061,4
2011-05-31,-0.007405,5


## Step 4: Save Results

In [5]:
faber_returns.to_csv('faber_returns.csv')
print('Saved: faber_returns.csv')

Saved: faber_returns.csv
